# SkillReuse: 1-Click Fast Replication Notebook
### *MSR 2027 Mining Challenge & CMP263 (Machine Learning)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/podorize/skillreuse/blob/main/notebooks/reproduce_in_colab.ipynb)

This notebook provides a **1-click, rapid replication (< 45 seconds)** of all empirical results, models, tables, and publication figures presented in the paper:
> **Predicting Cross-Repository Reuse and Diffusion of AI Agent Skills: A Multimodal Empirical Study on GitHub**

---

## 1. Setup & Dependencies
Clone the repository and install required dependencies.

In [ ]:
# If running inside Google Colab, clone the repo and enter its root directory
import os, sys
if not os.path.exists("src/gitskills_reuse"):
    !git clone https://github.com/podorize/skillreuse.git
    %cd skillreuse

!pip install -q scikit-learn pandas scipy matplotlib seaborn

## 2. Execute Fast Replication (< 45 seconds)
Loads the preprocessed feature matrices (160k train, 40k test across 14,568 unseen repositories), fits the champion models (`structure_only`, `structure_and_repo`, `hybrid_screened`), applies the frozen decision threshold ($t^* = 0.21$), and evaluates conditional intensity diffusion.

In [ ]:
import sys
sys.path.insert(0, "src")

from gitskills_reuse.reproduce import reproduce_study

# Run complete study reproduction
results_df = reproduce_study()

## 3. Consolidated Test Results (Paper Table 3)
Compare default threshold ($t=0.50$) vs. frozen optimal threshold ($t^*=0.21$).

In [ ]:
import pandas as pd

display_cols = ["model", "weighting", "threshold", "roc_auc", "pr_auc", "f1", "recall", "balanced_accuracy", "brier_score"]
res_table = results_df[results_df["threshold_rule"].isin(["default_0.50", "frozen_optimal_0.21")]]
res_table[display_cols].reset_index(drop=True)

## 4. Visualizations: Test Curves & Confusion Matrix
Display publication-ready figures on the 40,000 blind test instances.

In [ ]:
from IPython.display import Image, display

print("Figure 5(a): ROC and Precision-Recall Curves on Untouched External Test Set:")
display(Image("reports/modeling/figures/final_test_roc_pr_curves.png"))

print("Figure 5(b): Confusion Matrix of the Champion Hybrid Model under Frozen Threshold (t* = 0.21):")
display(Image("reports/modeling/figures/final_test_confusion_matrix.png"))

## 5. Explainability & Knowledge Discovery: The Three Pillars of Reuse
Display top positive drivers of cross-repository reuse (Odds Ratios $e^\beta$).

In [ ]:
imp_path = "reports/modeling/ablation/hybrid_screened/feature_importances.csv"
if os.path.exists(imp_path):
    imp = pd.read_csv(imp_path)
    top_drivers = imp[imp["task"] == "binary_reuse"].head(15)[["feature", "odds_ratio"]]
    print("Top 15 Drivers of AI Agent Skill Reuse:")
    display(top_drivers)
    
    display(Image("reports/modeling/figures/hybrid_shap_top20.png"))